<a href="https://colab.research.google.com/github/rishi-latchmepersad/TinyML-Home-Weather-Forecasting/blob/main/machine_learning/baseline_vs_measurements.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Baseline vs measurement comparison

This Colab notebook loads `baseline_forecast.log` and matching measurement CSV files, plots baseline forecasts versus actual temperatures, and computes an approximate mean absolute error (MAE) over the overlapping time range. It is designed to be flexible so future baseline logs and measurement files can be compared without code changes.


## Requirements

- Place `baseline_forecast.log` and matching `measurements_YYYY-MM-DD.csv` files in the measurements directory.
- Adjust the configuration cell to choose the forecast horizon and date range.
- Run cells top to bottom.


In [ ]:

# Install dependencies (uncomment if running outside of Colab where pandas/matplotlib are missing)
# %pip install pandas matplotlib


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pathlib
from datetime import timedelta

pd.set_option("display.max_rows", 10)

# Path to the measurements directory. Update this if your data lives elsewhere.
DATA_DIR = pathlib.Path("../measurements") if pathlib.Path("../measurements").exists() else pathlib.Path("measurements")
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Could not find measurements directory at {DATA_DIR.resolve()}")

print(f"Using data directory: {DATA_DIR.resolve()}")

In [ ]:
# Discover available baseline forecast and measurement files
baseline_log_path = DATA_DIR / "baseline_forecast.log"
measurement_files = sorted(DATA_DIR.glob("measurements_*.csv"))

if not baseline_log_path.exists():
    raise FileNotFoundError(f"Could not find baseline log at {baseline_log_path}")
if not measurement_files:
    raise FileNotFoundError("No measurement files found.")

print(f"Using baseline log: {baseline_log_path.name}")
print("Available measurement files (dates):")
for path in measurement_files:
    print(f"- {path.name}")


## Configuration

Choose the baseline horizon and one or multiple measurement files. If measurement filenames are omitted, the notebook attempts to auto-select files based on the baseline log date range. You can also adjust the sensor/quantity filters and the time tolerance used when aligning baseline forecasts to measurements.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pathlib

pd.set_option("display.max_rows", 10)

BASELINE_LOG_FILENAME = "baseline_forecast.log"

# Provide explicit measurement filenames. Leave empty to auto-select by the baseline log date span.
MEASUREMENT_FILENAMES = ["measurements_2026-02-25.csv", "measurements_2026-02-26.csv"]

TARGET_SENSOR = "bme280"
TARGET_QUANTITY = "temperature_c"
FORECAST_HORIZON_MINUTES = 60  # Matches forecast_t+60m_c
MERGE_TOLERANCE_MINUTES = 15  # Maximum time difference (minutes) when pairing baseline/measurement rows.

# Optional: Set start and stop datetimes for plotting. Leave as None to plot all available data.
START_DATETIME_STR = "2026-02-25 19:00:00+00:00"
STOP_DATETIME_STR = "2026-02-27 12:00:00+00:00"

baseline_log_path = DATA_DIR / BASELINE_LOG_FILENAME
if not baseline_log_path.exists():
    raise FileNotFoundError(f"Could not find baseline log at {baseline_log_path}")

selected_measurement_paths = []
if MEASUREMENT_FILENAMES:
    selected_measurement_paths = [DATA_DIR / name for name in MEASUREMENT_FILENAMES]
else:
    baseline_raw = pd.read_csv(baseline_log_path, parse_dates=["timestamp_iso8601"])
    if baseline_raw.empty:
        raise ValueError("baseline_forecast.log has no rows.")
    min_day = baseline_raw["timestamp_iso8601"].min().date()
    max_day = baseline_raw["timestamp_iso8601"].max().date()
    for day in pd.date_range(min_day, max_day, freq="D"):
        candidate = DATA_DIR / f"measurements_{day.date().isoformat()}.csv"
        if candidate.exists():
            selected_measurement_paths.append(candidate)

if not selected_measurement_paths:
    raise FileNotFoundError("Could not find matching measurement files. Set MEASUREMENT_FILENAMES explicitly.")

missing_measurements = [path for path in selected_measurement_paths if not path.exists()]
if missing_measurements:
    missing_names = ", ".join(path.name for path in missing_measurements)
    raise FileNotFoundError(f"Missing measurement files: {missing_names}")

print(f"Selected baseline log: {baseline_log_path.name}")
print("Selected measurement files:")
for path in selected_measurement_paths:
    print(f"- {path.name}")


In [ ]:
# Helpers

def forecast_column_for_horizon(minutes: int) -> str:
    return f"forecast_t+{minutes}m_c"


forecast_column = forecast_column_for_horizon(FORECAST_HORIZON_MINUTES)

# Load baseline forecast data
baseline_raw = pd.read_csv(baseline_log_path, parse_dates=["timestamp_iso8601"])
if baseline_raw.empty:
    raise ValueError(f"Baseline log {baseline_log_path.name} has no rows.")
if forecast_column not in baseline_raw.columns:
    raise ValueError(
        f"Column '{forecast_column}' was not found in {baseline_log_path.name}. "
        f"Available horizon columns: {[c for c in baseline_raw.columns if c.startswith('forecast_t+')]}"
    )

baseline_df = baseline_raw.sort_values("timestamp_iso8601").rename(
    columns={"timestamp_iso8601": "timestamp", forecast_column: "predicted_temperature_c"}
)
baseline_df["source_file"] = baseline_log_path.name
retained_columns = ["timestamp", "predicted_temperature_c", "source_file"]
if "inference_time_ms" in baseline_df.columns:
    retained_columns.append("inference_time_ms")
baseline_df = baseline_df[retained_columns]

# Load measurement data and filter to the target sensor/quantity
measurement_frames = []
for path in selected_measurement_paths:
    raw = pd.read_csv(path, parse_dates=["timestamp_iso8601"])
    filtered = raw[
        (raw["sensor"] == TARGET_SENSOR)
        & (raw["quantity"] == TARGET_QUANTITY)
    ].copy()
    if filtered.empty:
        raise ValueError(
            f"No measurements found for sensor '{TARGET_SENSOR}' and quantity '{TARGET_QUANTITY}' in {path.name}."
        )
    filtered = filtered.sort_values("timestamp_iso8601").rename(
        columns={"timestamp_iso8601": "timestamp", "value": "measured_temperature_c"}
    )
    filtered["source_file"] = path.name
    measurement_frames.append(filtered[["timestamp", "measured_temperature_c", "source_file"]])

measurements_df = pd.concat(measurement_frames, ignore_index=True).sort_values("timestamp")

print(baseline_df.head())
print(measurements_df.head())


In [ ]:
# Align baseline forecasts with the nearest measurement within the tolerance window
merge_tolerance = pd.Timedelta(minutes=MERGE_TOLERANCE_MINUTES)
aligned_df = pd.merge_asof(
    baseline_df,
    measurements_df[["timestamp", "measured_temperature_c"]],
    on="timestamp",
    direction="nearest",
    tolerance=merge_tolerance,
)

aligned_df = aligned_df.dropna(subset=["measured_temperature_c"])
if aligned_df.empty:
    raise ValueError(
        "No overlapping baseline forecast/measurement pairs were found within the tolerance window. "
        "Try increasing MERGE_TOLERANCE_MINUTES or verifying the file selection."
    )

aligned_df["absolute_error"] = (aligned_df["predicted_temperature_c"] - aligned_df["measured_temperature_c"]).abs()
mae = aligned_df["absolute_error"].mean()

print(f"Paired rows: {len(aligned_df)}")
print(f"Baseline MAE: {mae:.3f} °C")

In [ ]:
# Plot baseline forecasts vs. measurements over time

plot_df = aligned_df.copy()
if START_DATETIME_STR:
    plot_df = plot_df[plot_df["timestamp"] >= pd.to_datetime(START_DATETIME_STR)]
if STOP_DATETIME_STR:
    plot_df = plot_df[plot_df["timestamp"] <= pd.to_datetime(STOP_DATETIME_STR)]

plt.figure(figsize=(12, 5))
plt.plot(plot_df["timestamp"], plot_df["measured_temperature_c"], label="Measured temperature", marker="o")
for source, group in plot_df.groupby("source_file"):
    plt.plot(group["timestamp"], group["predicted_temperature_c"], label=f"Baseline forecast ({source})", marker="x")
plt.title(f"Baseline ({forecast_column}) vs measurements")
plt.xlabel("Timestamp")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Plot baseline latency and method metadata from logged data (if available)

baseline_meta_df = baseline_raw.copy()
if START_DATETIME_STR:
    baseline_meta_df = baseline_meta_df[baseline_meta_df["timestamp_iso8601"] >= pd.to_datetime(START_DATETIME_STR)]
if STOP_DATETIME_STR:
    baseline_meta_df = baseline_meta_df[baseline_meta_df["timestamp_iso8601"] <= pd.to_datetime(STOP_DATETIME_STR)]

if "method" in baseline_meta_df.columns:
    method_counts = baseline_meta_df["method"].value_counts()
    print("Baseline methods found:")
    print(method_counts)
else:
    print("No method column found; skipping baseline metadata summary.")

if "inference_time_ms" in baseline_meta_df.columns:
    latency_df = baseline_meta_df.dropna(subset=["inference_time_ms"]).copy()
    latency_df["inference_time_ms"] = pd.to_numeric(latency_df["inference_time_ms"], errors="coerce")
    latency_df = latency_df.dropna(subset=["inference_time_ms"]).sort_values("timestamp_iso8601")

    if latency_df.empty:
        print("No valid baseline inference_time_ms values found after filtering; skipping latency plot.")
    else:
        print("Baseline latency summary (ms):")
        print(latency_df["inference_time_ms"].describe())

        plt.figure(figsize=(12, 3))
        plt.plot(latency_df["timestamp_iso8601"], latency_df["inference_time_ms"], marker="o", color="tab:purple")
        plt.title("Baseline inference latency over time")
        plt.xlabel("Timestamp")
        plt.ylabel("Inference time (ms)")
        plt.grid(True)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No inference_time_ms column found; skipping baseline latency plot.")


## Error visualization

In [ ]:
# Optional: visualize the absolute error over time

plot_df = aligned_df.copy()
if START_DATETIME_STR:
    plot_df = plot_df[plot_df["timestamp"] >= pd.to_datetime(START_DATETIME_STR)]
if STOP_DATETIME_STR:
    plot_df = plot_df[plot_df["timestamp"] <= pd.to_datetime(STOP_DATETIME_STR)]

plt.figure(figsize=(12, 3))
plt.plot(plot_df["timestamp"], plot_df["absolute_error"], label="Absolute error", color="tab:red")
plt.title("Baseline forecast error over time")
plt.xlabel("Timestamp")
plt.ylabel("|Baseline - Measurement| (°C)")
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Saving aligned data (optional)

Uncomment the cell below to save the aligned baseline forecasts, measurements, and error values for further analysis.


In [ ]:
# output_path = DATA_DIR / f"aligned_baseline_{FORECAST_HORIZON_MINUTES}m.csv"
# aligned_df.to_csv(output_path, index=False)
# print(f"Saved aligned data to {output_path}")
